# MQAR: the paper's certified comparison

This notebook runs **the** MQAR experiment behind the paper's table. There is one
implementation (`paper/mqar/{data,model,train}.py`) and one driver
(`python -m paper.mqar.paper_sweep`); every number comes from that command and
nothing else.

## Protocol (fixed in `paper_sweep.PROTOCOL`)

| | |
|---|---|
| task | seq_len 512, 64 key-value pairs, vocab 8192, blank non-query slots |
| data | 100k train / 3k test, fixed train set, seed-disjoint test |
| model | depth 2, uniform layout (every layer is the mixer, no BaseConv), head_dim 64, position embeddings for attention-family mixers only, bf16 |
| optim | AdamW wd 0.1, cosine anneal, no warmup, no grad clipping, batch 128, 64 epochs, early stop at 99% recall |
| sweep | d in {64,128,256,512} x lr in logspace(-4,-2,4); report best-over-LR per (method, d) |
| methods | sdpa, linear_attention, nystrom_reference, flash_nystrom, flash_nystrom_tc, hyena, mamba |

This is Zoology's figure-2 recipe (Arora et al., ICLR 2024) at the (512, 64)
setting. **The harness is externally validated under it**: our Hyena reproduces
the DeltaNet paper's Figure 4 (18.6% vs their ~20% at d=512) and our Mamba
reproduces its ~100% (99.1% at d=256, arXiv:2406.06484).

## Cost and resuming

112 runs. Solvers early-stop at 99%; unsolved cells (Hyena, low-dim linear
attention) run all 64 epochs and dominate: budget on the order of a day on one
A100 at `--max_parallel 4`. The sweep is **resumable**: finished runs are skipped
by their result file, so if the session resets just re-run the sweep cell.
`--seeds 0 1 2` adds error bars at 3x cost; `--methods ...` trims the set.

In [ ]:
import torch
print(torch.__version__, "| cuda", torch.version.cuda, "| avail", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, "| cc", torch.cuda.get_device_capability(), "| mem_GB", round(p.total_memory/1e9, 1))

In [ ]:
# Setup (~35 min on a fresh runtime: flash_nystrom build + mamba-ssm source build).
import os, torch
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!git log --oneline -1
!pip -q install einops ninja packaging
CC = torch.cuda.get_device_capability()
assert CC >= (8, 0), "flash_nystrom + mamba-ssm need compute capability >= 8.0 (A100/L4/H100)"
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC[0]}.{CC[1]}"
os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"
!pip install -e . --no-build-isolation 2>&1 | tail -2
# Mamba CUDA kernels: setup.py imports torch, so build isolation must be off.
# On a bleeding-edge Colab torch there is no wheel and this compiles ~25 min.
!pip install causal-conv1d --no-build-isolation 2>&1 | tail -2
!pip install mamba-ssm --no-build-isolation 2>&1 | tail -2
import flash_nystrom
from paper.mqar.baselines import _HAS_MAMBA_CUDA
print("flash_nystrom", flash_nystrom.__version__, "| mamba CUDA kernels:", _HAS_MAMBA_CUDA)

In [ ]:
# THE sweep. Resumable: re-run this cell after any disconnect and it continues.
# Error bars: add --seeds 0 1 2 (3x cost). Trim: --methods sdpa flash_nystrom ...
!cd /content/FlashNystrom && python -u -m paper.mqar.paper_sweep --max_parallel 4

In [ ]:
# Re-print the table + summary.json from whatever has finished (safe anytime,
# also while the sweep is still running in another session).
!cd /content/FlashNystrom && python -m paper.mqar.paper_sweep --collect_only